# Public Notebook Skeleton

This notebook demonstrates how to run inference using the fine-tuned LoRA adapter on `Nemotron-3-Nano-30B-A3B-BF16` for the NVIDIA Nemotron Model Reasoning Challenge.

Requirements:
- `vllm` for fast inference
- `pandas` for data handling
- The base model and the LoRA adapter paths.

In [ ]:
import os
import json
import re
import pandas as pd
from tqdm import tqdm
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

## Configuration

In [ ]:
# Paths to the model and adapter
MODEL_PATH = "/kaggle/input/nvidia/nemotron-3-nano-30b-a3b-bf16" # Placeholder path
ADAPTER_PATH = "/kaggle/input/nemotron-reasoning-adapter/lora-rank-32" # Placeholder path
DATA_PATH = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/test.csv" # Placeholder path
OUTPUT_PATH = "submission.csv"

# Generation config
MAX_MODEL_LEN = 8192
MAX_LORA_RANK = 32
MAX_TOKENS = 7680
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_NUM_SEQS = 64
GPU_MEMORY_UTILIZATION = 0.85

## Initialize Engine

In [ ]:
# Initialize vLLM Offline inference Engine
try:
    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=MAX_NUM_SEQS,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        dtype="auto",
        max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=MAX_LORA_RANK,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )
except Exception as e:
    print(f"Skipping engine initialization locally: {e}")
    llm = None

sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
)

if llm is not None:
    lora_request = LoRARequest("adapter", 1, ADAPTER_PATH)
else:
    lora_request = None

## Answer Extraction

In [ ]:
def extract_answer(text):
    """
    Extracts the final answer from the model response enclosed in \boxed{}.
    """
    pattern = r'\\boxed\{([^}]*)\}'
    matches = re.findall(pattern, text)
    if matches:
        return matches[-1]
    return ""


## Inference Loop

In [ ]:
def run_inference():
    # Create mock data if test file doesn't exist
    if not os.path.exists(DATA_PATH):
        print("Mocking data for local testing...")
        df = pd.DataFrame({
            'id': [1, 2],
            'problem': ['Solve for x: 2x = 4', 'Convert 5 km to m']
        })
    else:
        df = pd.read_csv(DATA_PATH)

    prompts = [f"Problem:\n{prob}\n\nSolution:\n" for prob in df['problem']]
    
    if llm is not None:
        outputs = llm.generate(
            prompts,
            sampling_params=sampling_params,
            lora_request=lora_request,
        )
        
        answers = []
        for output in outputs:
            generated_text = output.outputs[0].text
            answers.append(extract_answer(generated_text))
    else:
        print("LLM not loaded, generating mock answers.")
        answers = ["2", "5000"]
    
    df['answer'] = answers
    df[['id', 'answer']].to_csv(OUTPUT_PATH, index=False)
    print(f"Saved to {OUTPUT_PATH}")

run_inference()